# Aula 8 — The electron density in real space, and the final project

Today has three parts:

1. **Hands-on: the electron density of silicon in real space.** The calculations are given; you plot the density and the orbitals on a plane and in 3D, and read the chemical bond from them.
2. **Questions on DFT** (graded): short answers, in your own words.
3. **Final project** (graded): a full study of one material, from an input card you write yourself to the question *metal or insulator?*

**First thing:** if you opened this notebook from the GitHub link, do File → Save a copy in Drive now, and work on that copy. Otherwise your answers are lost when the session ends.

**What to deliver:** this notebook, filled in and run, as an `.ipynb` file (File → Download → Download .ipynb).
**Deadline:** two weeks after Aula 8. Send it to the instructor as indicated in class.

All calculations here were checked with QE 6.7, the version `apt` installs on Colab.

## 1. Set up (every session)
Two cells: install QE (about 1 min), then get the course files from GitHub into your Drive.

The course files are on GitHub again: https://github.com/flavianojs/dft-cbpf. The second cell downloads the latest version
and adds to your `dft-cbpf` folder in My Drive **only the files you do not have yet** (this notebook, the new pseudopotentials).
If you have no `dft-cbpf` folder, it creates it.
Files that already exist are never overwritten: your notebooks and calculations of Aulas 6 and 7 stay as they are.

In [ ]:
# If an earlier attempt in this runtime left a broken fix behind, remove it first
# (otherwise every command prints "ld.so: object ... cannot be preloaded").
import os
if os.path.exists("/etc/ld.so.preload"):
    os.remove("/etc/ld.so.preload")

# Quantum ESPRESSO from Ubuntu's package manager (about 1 min).
!apt-get update -qq
!apt-get install -y -qq quantum-espresso > /dev/null

# Ubuntu 24.04 fix (see Aula 6): without it pw.x stops with "*** buffer overflow detected ***".
open("/tmp/qe_fix.c", "w").write(r"""
#include <stdio.h>
#include <stdarg.h>
/* Checked snprintf that truncates instead of aborting (false alarm of _FORTIFY_SOURCE). */
int __snprintf_chk(char *s, size_t maxlen, int flag, size_t slen, const char *fmt, ...)
{
    va_list ap;
    va_start(ap, fmt);
    int r = vsnprintf(s, maxlen < slen ? maxlen : slen, fmt, ap);
    va_end(ap);
    return r;
}
""")
# The fix is activated only if it compiled.
!gcc -O2 -shared -fPIC -o /usr/local/lib/qe_fix.so /tmp/qe_fix.c && echo /usr/local/lib/qe_fix.so > /etc/ld.so.preload
!cat /etc/ld.so.preload    # expected: /usr/local/lib/qe_fix.so

# Programs used today. Expected output: five paths in /usr/bin.
!which pw.x pp.x bands.x dos.x projwfc.x

# py3Dmol: 3D views of the density in the notebook.
!pip install -q py3Dmol

In [ ]:
# Mount Google Drive (authorize when asked) and go to the course folder.
from google.colab import drive
drive.mount("/content/drive")
%cd /content/drive/MyDrive

import glob, os, shutil, subprocess
REPO = "https://github.com/flavianojs/dft-cbpf"
TMP = "/content/dft-cbpf-latest"

def add_missing(src, dst):
    """Copy from src to dst only the files dst does not have. Nothing is overwritten."""
    n = 0
    for root, dirs, files in os.walk(src):
        dirs[:] = [d for d in dirs if d != ".git"]
        for f in files:
            target = os.path.join(dst, os.path.relpath(os.path.join(root, f), src))
            if not os.path.exists(target):
                os.makedirs(os.path.dirname(target), exist_ok=True)
                shutil.copy2(os.path.join(root, f), target)
                n += 1
    print(n, "new files added to", dst)

shutil.rmtree(TMP, ignore_errors=True)
ok = subprocess.run(["git", "clone", "-q", "--depth", "1", REPO, TMP]).returncode == 0
if ok:
    add_missing(TMP, "dft-cbpf")
else:
    # Plan B, if GitHub cannot be reached: the dft-cbpf.zip given in class, at the top of My Drive.
    zips = sorted(glob.glob("*dft-cbpf*.zip"), key=os.path.getmtime)
    if zips:
        print("GitHub not reachable; using", zips[-1])
        !unzip -n -q "{zips[-1]}"
    elif not os.path.isdir("dft-cbpf"):
        raise RuntimeError("GitHub not reachable and no dft-cbpf.zip in My Drive: ask the instructor for the zip.")

# Today's files must be there.
for f in ["src/silicon/pw.scf.silicon.in", "src/pseudos/Si.pz-vbc.UPF",
          "src/pseudos/Al.pbe-n-rrkjus_psl.1.0.0.UPF", "src/pseudos/C.pbe-n-rrkjus_psl.1.0.0.UPF",
          "src/pseudos/Mg.pbe-spnl-rrkjus_psl.1.0.0.UPF", "src/pseudos/O.pbe-n-rrkjus_psl.1.0.0.UPF"]:
    assert os.path.exists("dft-cbpf/" + f), "missing: dft-cbpf/" + f

%cd dft-cbpf/calculations
!ls ../src/pseudos

## 2. Helper functions
The same as in Aula 7, plus `efermi`, which reads the Fermi energy from an output. Everything goes to `calculations/aula8/`.

In [ ]:
import glob, re, subprocess, time
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt


def set_var(text, name, value):
    # Change the line  name = ...  of a namelist (first match, any case). See Aula 6.
    pat = rf"^(\s*{re.escape(name)}\s*=\s*).*$"
    text, n = re.subn(pat, rf"\g<1>{value}", text, count=1, flags=re.M | re.I)
    assert n == 1, f"{name} not found"
    return text


def add_var(text, name, value, namelist="&SYSTEM"):
    # Insert  name = value  right after the line that opens the namelist (for new variables).
    text, n = re.subn(rf"^(\s*{namelist}.*)$", rf"\g<1>\n  {name} = {value}", text,
                      count=1, flags=re.M | re.I)
    assert n == 1, f"{namelist} not found"
    return text


def set_kpts(text, mesh):
    # Replace the line after K_POINTS, e.g. "8 8 8 1 1 1". See Aula 6.
    return re.sub(r"(K_POINTS.*\n).*", rf"\g<1>  {mesh}", text, count=1)


def run(text, name, prog="pw.x", workdir="aula8"):
    # Write workdir/name.in, run   prog < name.in > name.out   inside workdir, return the output.
    w = Path(workdir)
    w.mkdir(exist_ok=True)
    (w / f"{name}.in").write_text(text)
    t0 = time.time()
    with open(w / f"{name}.in") as inp, open(w / f"{name}.out", "w") as out:
        subprocess.run([prog], stdin=inp, stdout=out, stderr=subprocess.STDOUT, cwd=w)
    out = (w / f"{name}.out").read_text()
    if "JOB DONE" not in out:
        print(out[-1500:])
        raise RuntimeError(f"{prog} failed: see {workdir}/{name}.out")
    print(f"{prog:10s} {name:16s} done in {time.time() - t0:5.1f} s", flush=True)
    return out


def etot(out):
    # Converged total energy in Ry (the line that starts with "!").
    return float(re.findall(r"!\s+total energy\s+=\s+(\S+)", out)[-1])


def efermi(out):
    # Fermi energy in eV, as printed by pw.x for metals ("the Fermi energy is ...").
    return float(re.findall(r"the Fermi energy is\s+(\S+)", out)[-1])


RY_EV = 13.605693   # 1 Ry in eV

The next cell defines the tools for the electron density in real space (section 3 and the project): they write the `pp.x` inputs for you.
You do not need to read them now; each one starts with a comment that says what it does.

In [ ]:
FCC = np.array([[0, 0, 0], [0, .5, .5], [.5, 0, .5], [.5, .5, 0]])   # fcc sites of the cubic cell (alat units)
Z = {"H": 1, "C": 6, "O": 8, "Mg": 12, "Al": 13, "Si": 14, "Cu": 29}


def pp_extract(prefix, filplot, plot_num, workdir="aula8", **extra):
    # First stage of pp.x: extract a quantity (plot_num) from the files of prefix into filplot.
    # extra: more &INPUTPP variables, e.g. kpoint=1, kband=2, lsign=".true."
    more = "".join(f"  {k} = {v}\n" for k, v in extra.items())
    text = f"&INPUTPP\n  prefix = '{prefix}', outdir = './tmp/'\n  filplot = '{filplot}', plot_num = {plot_num}\n{more}/\n"
    run(text, "pp_" + filplot, prog="pp.x", workdir=workdir)


def plane110(filplot, workdir="aula8", nx=113, ny=80):
    # Second stage of pp.x: the quantity in filplot on the (110) plane through the origin,
    # spanned by e1 = [1,1,0] and e2 = [0,0,1] (alat units), starting at x0 = (0, 0, -0.375).
    # For the fcc, diamond and rock-salt structures this plane contains the nearest-neighbour bonds.
    # Returns X, Y (in bohr) and F, 2D arrays ready for plt.contourf(X, Y, F).
    out = filplot + "_110.dat"
    text = f"""&INPUTPP
/
&PLOT
  nfile = 1, filepp(1) = '{filplot}', weight(1) = 1.0
  iflag = 2, output_format = 7, fileout = '{out}'
  e1(1) = 1.0, e1(2) = 1.0, e1(3) = 0.0
  e2(1) = 0.0, e2(2) = 0.0, e2(3) = 1.0
  x0(1) = 0.0, x0(2) = 0.0, x0(3) = -0.375
  nx = {nx}, ny = {ny}
/
"""
    run(text, "pp2d_" + filplot, prog="pp.x", workdir=workdir)
    d = np.loadtxt(Path(workdir) / out)
    X, Y, F = (d[:, i].reshape(nx, ny) for i in range(3))
    return X, Y, F


def atoms110(alat, positions):
    # Atoms in the window of plane110, as (x, y) in bohr, with their symbols.
    # positions: list of (symbol, (x, y, z)) in alat units, as in ATOMIC_POSITIONS (alat).
    xs, ys, names = [], [], []
    for s, p in positions:
        for t in FCC:
            for n in np.ndindex(5, 5, 5):
                r = np.array(p) + t + np.array(n) - 2
                if abs(r[0] - r[1]) < 1e-6 and -1e-6 <= r[0] <= 1 + 1e-6 and -0.375 - 1e-6 <= r[2] <= 0.625 + 1e-6:
                    xs.append(np.sqrt(2) * r[0] * alat); ys.append((r[2] + 0.375) * alat); names.append(s)
    return np.array(xs), np.array(ys), names


def cube_box(filplot, alat, positions, workdir="aula8", n=48):
    # Second stage of pp.x: the quantity in filplot on the cubic cell (side alat), written as a
    # Gaussian .cube file with all the atoms of the cubic cell, for show3d. Returns the file name.
    raw = filplot + "_box.xsf"
    text = f"""&INPUTPP
/
&PLOT
  nfile = 1, filepp(1) = '{filplot}', weight(1) = 1.0
  iflag = 3, output_format = 3, fileout = '{raw}'
  e1(1) = 1.0, e1(2) = 0.0, e1(3) = 0.0
  e2(1) = 0.0, e2(2) = 1.0, e2(3) = 0.0
  e3(1) = 0.0, e3(2) = 0.0, e3(3) = 1.0
  x0(1) = 0.0, x0(2) = 0.0, x0(3) = 0.0
  nx = {n}, ny = {n}, nz = {n}
/
"""
    run(text, "pp3d_" + filplot, prog="pp.x", workdir=workdir)
    lines = (Path(workdir) / raw).read_text().splitlines()
    i = lines.index("DATAGRID_3D_UNKNOWN") + 6
    vals = " ".join(l for l in lines[i:] if not l.strip().startswith("END")).split()
    data = np.array(vals, float).reshape(n, n, n)          # first index runs fastest in the file
    atoms = []
    for s, p in positions:
        for t in FCC:
            for m in np.ndindex(3, 3, 3):
                r = (np.array(p) + t + np.array(m) - 1)
                if np.all(r > -1e-6) and np.all(r < 1 + 1e-6):
                    atoms.append((s, r * alat))
    h = alat / n
    head = ["pp.x " + filplot, "cubic cell", f"{len(atoms):5d} 0.0 0.0 0.0",
            f"{n:5d} {h:.6f} 0.0 0.0", f"{n:5d} 0.0 {h:.6f} 0.0", f"{n:5d} 0.0 0.0 {h:.6f}"]
    head += [f"{Z[s]:5d} {float(Z[s]):.1f} {r[0]:.6f} {r[1]:.6f} {r[2]:.6f}" for s, r in atoms]
    body = data.transpose(2, 1, 0).reshape(-1)              # cube files: last index runs fastest
    cube = Path(workdir) / (filplot + ".cube")
    cube.write_text("\n".join(head) + "\n" + "\n".join(" ".join(f"{v:.5e}" for v in body[k:k + 6])
                                                      for k in range(0, len(body), 6)) + "\n")
    return str(cube)


def read_cube(cube):
    # Read a .cube file: returns the 3D array of values and the volume of one grid cell (bohr^3).
    lines = Path(cube).read_text().splitlines()
    nat = int(lines[2].split()[0])
    axes = np.array([[float(v) for v in lines[i].split()] for i in (3, 4, 5)])
    n, vec = axes[:, 0].astype(int), axes[:, 1:]
    data = np.array(" ".join(lines[6 + nat:]).split(), float).reshape(n)
    return data, abs(np.linalg.det(vec))


def show3d(cube, iso, signed=False, width=500, height=450):
    # Isosurface of a .cube file with py3Dmol (rotate with the mouse, zoom with the wheel).
    # signed=True: for psi from lsign (|psi|^2 times the sign of psi): +iso in red, -iso in blue.
    import py3Dmol
    text = Path(cube).read_text()
    v = py3Dmol.view(width=width, height=height)
    v.addModel(text, "cube")
    v.setStyle({"sphere": {"scale": 0.25, "colorscheme": "Jmol"}})
    v.addVolumetricData(text, "cube", {"isoval": iso, "color": "#d62728", "opacity": 0.85})
    if signed:
        v.addVolumetricData(text, "cube", {"isoval": -iso, "color": "#1f77b4", "opacity": 0.85})
    v.zoomTo()
    return v.show()

## 3. Hands-on: the electron density of silicon in real space

So far we looked at the electrons in **energy** (bands, DOS). Today we look at them in **space**.
The Kohn–Sham density is built from the orbitals:

$$ n(\mathbf r) = 2 \sum_{n\mathbf k}^{\text{occ}} w_{\mathbf k}\, |\psi_{n\mathbf k}(\mathbf r)|^2 $$

(the 2 is the spin). `pw.x` keeps n(r) and the ψ's in its `outdir`; the post-processing program **`pp.x`** reads them and writes them on a grid you can plot. It works in two stages:

1. `&INPUTPP`: **which quantity** (`plot_num`). `plot_num = 0` is the valence density n(r); `plot_num = 7` is |ψ_nk(r)|² of one band `kband` at one k point `kpoint`
   (with `lsign = .true.` at Γ, |ψ|² times the sign of ψ, so that you can see where ψ changes sign). It writes the quantity on the FFT grid to a file (`filplot`).
2. `&PLOT`: **where**. `iflag = 2` interpolates it on a plane, `iflag = 3` on a 3D box.

The helper functions of section 2 write these inputs for you: `pp_extract` (stage 1), `plane110` and `cube_box` (stage 2), `show3d` (3D view).

We use the silicon of Aulas 6 and 7 (input `src/silicon/pw.scf.silicon.in`: LDA, 30 Ry, 6×6×6 mesh). Two runs of `pw.x`:
the `scf` (the density), then a `bands` run with **only the Γ point**, for the orbitals at Γ. Both take a few seconds.

In [ ]:
si = Path("../src/silicon/pw.scf.silicon.in").read_text()
si = set_var(si, "pseudo_dir", "'../../src/pseudos'")   # relative to aula8/
out = run(si, "si_scf")
print(f"E = {etot(out):.8f} Ry")

# The orbitals at Gamma: a 'bands' run with a single k point (same prefix: it reads the scf density).
gam = set_var(si, "calculation", "'bands'")
gam = re.sub(r"K_POINTS.*\n.*", "K_POINTS tpiba\n1\n  0.0 0.0 0.0 1.0", gam)
out_g = run(gam, "si_gamma")
eig_g = [float(v) for v in out_g.split("bands (ev):")[1].split("Writing")[0].split()]
print("eigenvalues at Gamma (eV):", eig_g)

SI_A = 10.26                                        # celldm(1), bohr
SI_POS = [("Si", (0.0, 0.0, 0.0)), ("Si", (0.25, 0.25, 0.25))]   # ATOMIC_POSITIONS (alat)

### 3.1 Exercise A: the density on a plane
The plane of `plane110` is a (110) plane: it contains the zig-zag chains of Si–Si bonds. The window is one cell long along [110] (14.5 bohr) and one cell high along [001] (10.26 bohr).

Complete the cell: a filled contour plot of n(r) with `plt.contourf(X, Y, F, 30)`, a color bar, and the atoms (white dots) on top.

In [ ]:
pp_extract("silicon", "si_rho", 0)          # stage 1: plot_num = 0, the valence density
X, Y, F = plane110("si_rho")                # stage 2: on the (110) plane; X, Y in bohr, F in electrons/bohr^3
xa, ya, _ = atoms110(SI_A, SI_POS)          # the atoms in that window

plt.figure(figsize=(9, 6))
plt.contourf(..., ..., ..., 30, cmap="viridis")
plt.colorbar(label="n (electrons/bohr$^3$)")
plt.plot(..., ..., "wo")
plt.gca().set_aspect("equal")
plt.xlabel("along [110] (bohr)"); plt.ylabel("along [001] (bohr)")
plt.title("Si: valence density on the (110) plane")
plt.show()

1. Where is the density largest? What does this tell you about the chemical bond in silicon?
2. Why is the density almost **zero at the nuclei**? (Think about what the pseudopotential does with the 1s, 2s and 2p electrons, and about the shape of the pseudo-wavefunctions near the nucleus.)

**Your answer:**

_(write here)_

### 3.2 Exercise B: count the electrons, and look in 3D
`cube_box` writes the density on the **cubic** cell of side a (a .cube file, the format most 3D viewers read), and `read_cube` reads it back:
it returns the values on the grid and the volume `dV` of one grid cell.

1. Integrate n(r) over the cubic cell: N = Σ n · dV. How many electrons do you find? Explain the number (how many primitive cells, atoms and valence electrons per atom are in the cubic cell?).
2. Run the 3D view (drag to rotate, wheel to zoom). Try isovalues between 0.03 and 0.08. What does the isosurface look like at 0.06?

In [ ]:
cube = cube_box("si_rho", SI_A, SI_POS)      # aula8/si_rho.cube
n3d, dV = read_cube(cube)
N = ...
print(f"electrons in the cubic cell: {N:.3f}")

In [ ]:
show3d("aula8/si_rho.cube", 0.06)

**Your answer:**

_(write here)_

### 3.3 Exercise C: the orbitals at Γ
The eigenvalues at Γ printed above come in groups: band 1, bands 2–4 (degenerate: the **top of the valence band**), bands 5–7 (degenerate: the **bottom of the conduction band** at Γ), band 8.

For a degenerate group, any combination of the states is also an eigenstate: a single member is arbitrary, the **sum of |ψ|² over the group** is not.

1. Complete the cell: for each group, `F` is the sum over its bands of |ψ|². (The files have |ψ|² times the sign of ψ: take the absolute value.)
2. Where is the charge of the top of the valence band (bands 2–4), and where is the charge of the bottom of the conduction band (bands 5–7), **relative to the bonds**? Which one is **bonding**, which one **antibonding**?
3. What happens to the bonds when light excites an electron across the gap?
4. Look at ψ itself in 3D, with its sign (red +, blue −), for band 2 and for band 5. In which one does ψ change sign **between** two bonded atoms?

In [ ]:
for b in range(1, 9):     # |psi|^2 (with the sign of psi) of the 8 bands at Gamma (kpoint 1)
    pp_extract("silicon", f"si_psi{b}", 7, kpoint=1, kband=b, lsign=".true.")
maps = {b: plane110(f"si_psi{b}")[2] for b in range(1, 9)}

groups = {"band 1": [1], "bands 2-4 (top of valence)": [2, 3, 4], "bands 5-7 (bottom of conduction)": [5, 6, 7]}
fig, axs = plt.subplots(1, 3, figsize=(17, 4.5))
for ax, (title, bs) in zip(axs, groups.items()):
    F = ...
    ax.contourf(X, Y, F, 30, cmap="viridis")
    ax.plot(xa, ya, "wo")
    ax.set_aspect("equal"); ax.set_title(title)
plt.show()

In [ ]:
show3d(cube_box("si_psi2", SI_A, SI_POS), 0.004, signed=True)

In [ ]:
show3d(cube_box("si_psi5", SI_A, SI_POS), 0.004, signed=True)

**Your answer:**

_(write here)_

## 4. Questions on DFT (graded)
Answer in a few sentences each, in your own words; formulas are welcome. You may use your notes and the lecture notes of the course.
Where a question refers to a calculation, use the numbers you obtained in the hands-ons.

**Q1. Hohenberg–Kohn.** What does the first Hohenberg–Kohn theorem state? Why does it allow us to work with the density n(r), a function of 3 variables, instead of the many-body wavefunction?

**Your answer:**

_(write here)_

**Q2. Kohn–Sham.** What is the Kohn–Sham system? Which part of the energy is treated exactly, and which part has to be approximated?

**Your answer:**

_(write here)_

**Q3. Exchange and correlation.** What is the difference between LDA and GGA (for example PBE)? Which tends to give smaller lattice parameters, and why does that matter for your project?

**Your answer:**

_(write here)_

**Q4. Plane waves.** What does `ecutwfc` control? Why does the total energy always decrease when you increase `ecutwfc`, while it can go up or down when you increase the k-point mesh?

**Your answer:**

_(write here)_

**Q5. Pseudopotentials.** Why do we use pseudopotentials? Why do ultrasoft pseudopotentials need `ecutrho` larger than 4 × `ecutwfc` (we use 8×)? What does `spnl` in `Mg.pbe-spnl-rrkjus_psl.1.0.0.UPF` tell you about the valence?

**Your answer:**

_(write here)_

**Q6. Metals and smearing.** Why does a metal need more k points than an insulator? What does `degauss` do, and what goes wrong if it is too small or too large?

**Your answer:**

_(write here)_

**Q7. The SCF cycle.** Why does pw.x not simply use the output density as the next input density? What does `mixing_beta` control?

**Your answer:**

_(write here)_

**Q8. scf and nscf.** Why do we compute the DOS with an `nscf` run on a denser mesh instead of the `scf` mesh? Why are tetrahedra preferred to smearing for a DOS?

**Your answer:**

_(write here)_

**Q9. Kohn–Sham eigenvalues and the gap.** Can the Kohn–Sham gap ε_LUMO − ε_HOMO be compared with the experimental gap? Explain, using what you know about LDA/GGA and the silicon result of Aula 7 (0.52 eV against 1.17 eV).

**Your answer:**

_(write here)_

**Q10. Magnetism.** State the Stoner criterion. Use it, with your numbers of Aula 7, to explain why bcc iron is ferromagnetic. Copper has 11 valence electrons and its d band is full, about 2 eV below E_F: would you expect it to be magnetic? Why?

**Your answer:**

_(write here)_

## 5. Final project (graded)

### 5.0 Your material
Each student studies **one** of the three materials below, assigned by the instructor. Write it in the next cell.

| material | structure | experimental a (Å) | atoms (alat units) | pseudopotentials in `src/pseudos/` |
| --- | --- | --- | --- | --- |
| **Al** | fcc | 4.050 | Al (0, 0, 0) | `Al.pbe-n-rrkjus_psl.1.0.0.UPF` |
| **C** (diamond) | diamond (fcc lattice, 2 atoms) | 3.567 | C (0, 0, 0), C (¼, ¼, ¼) | `C.pbe-n-rrkjus_psl.1.0.0.UPF` |
| **MgO** | rock salt (fcc lattice, 2 atoms) | 4.212 | Mg (0, 0, 0), O (½, ½, ½) | `Mg.pbe-spnl-rrkjus_psl.1.0.0.UPF`, `O.pbe-n-rrkjus_psl.1.0.0.UPF` |

All three have an **fcc lattice** (`ibrav = 2`), like silicon, so the Brillouin zone and the band path of Aula 7 apply.
The pseudopotentials are PBE and ultrasoft. **Read the header of each one** (`!head -40 ../src/pseudos/...`): number of valence electrons and suggested cutoffs.

**What is evaluated** (suggested weights):

| part | weight |
| --- | --- |
| Questions on DFT (section 4) | 25% |
| 5.1 Input card | 10% |
| 5.2 Convergence tests | 15% |
| 5.3 Lattice parameter | 10% |
| 5.4–5.5 Bands, DOS and PDOS | 15% |
| 5.6 Electron density in real space | 10% |
| 5.7 Metal or insulator? Discussion | 10% |
| 5.8 Reproducibility | 5% |

Every number you report must come from a calculation in this notebook, and every choice (cutoff, mesh, smearing) must be justified.
Plots need axis labels with units. A short, clear justification is worth more than a long one.

In [ ]:
MATERIAL = "..."     # "Al", "C" or "MgO": the one assigned to you
PROJ = f"project_{MATERIAL}"   # all project files go to calculations/project_<material>/

### 5.1 The input card
Write, **from scratch**, the `scf` input for your material at the experimental lattice parameter. Use the silicon input of Aulas 6 and 7 (`../src/silicon/pw.scf.silicon.in`) and the iron input of Aula 7 (`../src/fe/pw.scf.fe.in`, ultrasoft, with smearing) as examples. Check list:

* `&CONTROL`: `calculation`, `prefix`, `outdir`, `pseudo_dir = '../../src/pseudos'`;
* `&SYSTEM`: `ibrav = 2`, `celldm(1)` **in bohr** (1 bohr = 0.529177 Å), `nat`, `ntyp`, `ecutwfc`, `ecutrho` (ultrasoft: 8 × `ecutwfc`);
* occupations: is your material a metal? If you do not know yet, what is the safe choice? (Explain in the cell below the input.)
* `ATOMIC_SPECIES`: element, mass, pseudopotential file; `ATOMIC_POSITIONS alat`; `K_POINTS automatic`.

Start with the cutoff suggested in the pseudopotential header and a modest mesh: the next section is where you converge them.

In [ ]:
my_input = """
&CONTROL
  ...
/
&SYSTEM
  ...
/
&ELECTRONS
  conv_thr = 1.0d-9
/
ATOMIC_SPECIES
  ...
ATOMIC_POSITIONS alat
  ...
K_POINTS automatic
  ...
"""
out = run(my_input, "scf_test", workdir=PROJ)
print(f"E = {etot(out):.8f} Ry")
!grep -E "number of electrons|number of k points|kinetic-energy cutoff|charge density cutoff" {PROJ}/scf_test.out

**Your choice of occupations, and why:**

_(write here)_

### 5.2 Convergence tests
Converge `ecutwfc` (with `ecutrho = 8 × ecutwfc`) and the k-point mesh, as in Aula 6.

1. **Choose a criterion** and state it, for example: total energy within 1 meV/atom of a much better calculation. (Optional, better: also check that the pressure or the lattice parameter is converged.)
2. Scan `ecutwfc` at a fixed, reasonable mesh. Plot the error against `ecutwfc` on a log scale.
3. Scan the mesh at the chosen cutoff. Try centered and shifted meshes. If your material is a metal, keep the smearing fixed while you do it, and comment on how smooth the convergence is.
4. Report your converged parameters and the time one `scf` takes with them.

If your criterion asks for a cutoff that makes the next sections too slow on Colab, say so, and show instead that the quantity you report (a₀, the gap) is converged at the cutoff you use. The k mesh and the cutoff converge almost independently, so you may scan the mesh at a moderate cutoff.

In [ ]:
# 5.2 (a): ecutwfc. Complete the loop; use set_var(text, "ecutwfc", ...) and set_var(text, "ecutrho", ...).
cutoffs = [...]
E_cut = []
for ec in cutoffs:
    ...

In [ ]:
# 5.2 (b): k-point mesh at the chosen cutoff. Use set_kpts(text, "n n n s s s").
meshes = [...]
E_k = []
for n in meshes:
    ...

**Criterion, converged ecutwfc and mesh, and why:**

_(write here)_

### 5.3 The lattice parameter
With the converged parameters, find the equilibrium lattice parameter a₀:

1. compute E(a) for 7 or more values of `celldm(1)` around the experimental one (±3% is a good range);
2. fit (cubic polynomial as in Aula 6, or a Birch–Murnaghan equation of state) and find a₀;
3. compare with experiment: percent error. Is it what you expect from PBE?
4. (Optional) check with a `vc-relax` (Aula 6), and/or give the bulk modulus.

**Use a₀ in all the following sections.**

In [ ]:
# 5.3: E(a) and the fit.
...

**a₀, comparison with experiment, comment:**

_(write here)_

### 5.4 Band structure
1. `scf` at a₀, then `bands` along the path L–Γ–X–U|K–Γ (the `K_POINTS crystal_b` card of Aula 7), then `bands.x`.
2. Plot the bands against the path, with the high-symmetry labels, and with the zero of energy at the Fermi level (metal) or at the top of the valence band (insulator).
3. Set `nbnd` so that you see some empty bands.

In [ ]:
# 5.4: scf at a0, bands, bands.x, plot.
...

### 5.5 DOS and PDOS
1. `nscf` on a denser mesh than the `scf` one, with `occupations = 'tetrahedra_opt'`; then `dos.x` and `projwfc.x`.
2. Plot the total DOS and the PDOS of each atom and orbital (s, p, and d if present), with the same zero of energy as the bands.
3. Check that the integrated DOS gives the number of valence electrons.
4. Which orbitals form the top of the valence band and the bottom of the conduction band (or the states at E_F)?

In [ ]:
# 5.5: nscf, dos.x, projwfc.x, plots.
...

**What the DOS and PDOS show:**

_(write here)_

### 5.6 The electron density in real space
As in section 3, with the tools of section 2 (`pp_extract`, `plane110`, `atoms110`, `cube_box`, `read_cube`, `show3d`). Everything at a₀, with your converged parameters.

1. Plot the valence density n(r) on the (110) plane of `plane110`, with the atoms. For all three materials this plane contains the nearest-neighbour bonds.
2. Integrate n(r) over the cubic cell (`cube_box`, `read_cube`): does it give the number you expect?
3. Run a `bands` calculation with only the Γ point and plot, on the same plane, the sum of |ψ|² over the **highest occupied state at Γ and the states degenerate with it** (for a metal: the highest state at Γ below E_F).
4. **What kind of bond?** From the maps: where is the charge, between the atoms, on one of them, or spread over the cell? Is the bond in your material covalent, ionic or metallic? Compare with silicon (section 3).

*Color scale:* if the density of one atom is much larger than the rest (semicore states in the pseudopotential, as the 2s2p of Mg), a linear scale hides everything else: use a logarithmic one: `from matplotlib.colors import LogNorm`, then `plt.contourf(X, Y, F, levels=np.logspace(np.log10(F.min()), np.log10(F.max()), 30), norm=LogNorm())`.

*Note:* `pp.x` reads the files of the last `pw.x` run with the same prefix. Run the Γ `bands` calculation after the density (`plot_num = 0` only needs the `scf`), or run a new `scf` at a₀ first.

In [ ]:
# 5.6: density and the highest occupied state(s) at Gamma, on the (110) plane.
...

**What kind of bond, and the evidence:**

_(write here)_

### 5.7 Metal or insulator?
Answer with evidence from **both** the bands and the DOS:

* Metal: which bands cross E_F, and what is the DOS at E_F?
* Insulator: the gap, its value, and whether it is direct or indirect (between which k points). Compare with the experimental gap (look it up and cite the source). Is the difference what you expect for PBE? Why (question Q9)?

**Your answer:**

_(write here)_

### 5.8 Reproducibility
Fill in the table, so that someone else could repeat your calculations exactly.

| item | value |
| --- | --- |
| Code and version | |
| Exchange-correlation functional | |
| Pseudopotentials (file names) | |
| ecutwfc / ecutrho (Ry) | |
| k mesh (scf) and shift | |
| k mesh (nscf, DOS) | |
| Occupations / smearing / degauss | |
| conv_thr | |
| Lattice parameter used (bohr and Å) | |
| Number of k points along the band path | |